In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_timestamp, concat_ws, lit, when, col

In [2]:

spark = SparkSession.builder \
    .appName("SilverLayer") \
    .config(
        "spark.jars.packages",
        "io.delta:delta-spark_2.12:3.1.0"
    ) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

26/06/20 16:06:55 WARN Utils: Your hostname, Saileshs-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.4 instead (on interface en0)
26/06/20 16:06:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Users/saileshpola/PycharmProjects/PythonProject/PysparkKafkaETE/.venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/saileshpola/.ivy2/cache
The jars for the packages stored in: /Users/saileshpola/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a8c652f0-00e9-4e8d-8674-8944827875c8;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 267ms :: artifacts dl 8ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     | 

In [3]:
df = spark.readStream.format("delta").load("../BronzeLayer/data/bronze/trades")

In [4]:
flattened_df = df.select(
    col("data.trade_id").alias("trade_id"),
    col("data.trader_id").alias("trader_id"),
    col("data.symbol").alias("symbol"),
    col("data.price").alias("price"),
    col("data.quantity").alias("quantity"),
    col("data.trade_timestamp").alias("trade_timestamp"),
    col("data.ingestion_timestamp").alias("ingestion_timestamp"),
    col("data.exchange").alias("exchange"),
    col("data.side").alias("side"),
    col("data.broker").alias("broker"),
    col("data.metadata.source").alias("source"),
    col("data.metadata.version").alias("version"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("kafka_timestamp"),
    col("ingestion_time")
)

In [5]:
flattened_df.printSchema()

root
 |-- trade_id: string (nullable = true)
 |-- trader_id: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- trade_timestamp: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- exchange: string (nullable = true)
 |-- side: string (nullable = true)
 |-- broker: string (nullable = true)
 |-- source: string (nullable = true)
 |-- version: integer (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)



In [6]:
parsed_ts = to_timestamp(col("trade_timestamp"))

trade_id_valid = col("trade_id").isNotNull()
trader_id_valid = col("trader_id").isNotNull()
price_type_valid = col("price").cast("double").isNotNull()
price_value_valid = col("price") > 0
quantity_type_valid = col("quantity").cast("int").isNotNull()
quantity_value_valid = col("quantity") > 0
trade_timestamp_valid = parsed_ts.isNotNull()
trade_timestamp_not_future = parsed_ts <= col("ingestion_time")
side_valid = col("side").isin(["BUY", "SELL"])
exchange_valid = col("exchange").isin(["NYSE", "NASDAQ", "BINANCE"])


In [7]:
valid_condition = (
    trade_id_valid &
    trader_id_valid &
    price_type_valid &
    price_value_valid &
    quantity_type_valid &
    quantity_value_valid &
    trade_timestamp_valid &
    trade_timestamp_not_future &
    side_valid &
    exchange_valid
)

valid_trades = flattened_df.filter(valid_condition)
dlq_trades = (flattened_df.filter(~valid_condition).withColumn("dlq_reason", concat_ws(
        ", ",
        when(~trade_id_valid, lit("missing_trade_id")),
        when(~trader_id_valid, lit("missing_trader_id")),
        when(~price_type_valid, lit("invalid_price_type")),
        when(price_type_valid & ~price_value_valid, lit("invalid_price_value")),
        when(~quantity_type_valid, lit("invalid_quantity_type")),
        when(quantity_type_valid & ~quantity_value_valid, lit("invalid_quantity_value")),
        when(~trade_timestamp_valid, lit("invalid_trade_timestamp")),
        when(trade_timestamp_valid & ~trade_timestamp_not_future, lit("future_trade_timestamp")),
        when(~side_valid, lit("invalid_side")),
        when(~exchange_valid, lit("invalid_exchange"))
    ))
              )

valid_trades = valid_trades.withColumn(
    "trade_timestamp",
    parsed_ts
)

In [8]:
valid_trades = valid_trades.withWatermark("trade_timestamp", "10 minutes").dropDuplicates(["trade_id"])

In [9]:
valid_query = (valid_trades.writeStream.format("delta")
        .outputMode("append")
        .option("checkpointLocation", 'checkpoints/silver/trades')
        .option("path", "data/silver/trades")
        .trigger(processingTime="10 seconds")
        .queryName("silver_trades")
        .start())

In [10]:
dlq_query = (dlq_trades.writeStream.format("delta")
        .outputMode("append")
        .option("checkpointLocation", 'checkpoints/dlq/trades')
        .option("path", "data/dlq/trades")
        .trigger(processingTime="10 seconds")
        .queryName("dlq_trades")
        .start())
spark.streams.awaitAnyTermination()

ERROR:root:KeyboardInterrupt while sending command.                 (0 + 0) / 3]
Traceback (most recent call last):
  File "/Users/saileshpola/PycharmProjects/PythonProject/PysparkKafkaETE/.venv/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/Users/saileshpola/PycharmProjects/PythonProject/PysparkKafkaETE/.venv/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/Users/saileshpola/.pyenv/versions/3.10.13/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 